# V11 Phase 0 #5 — MCMC vs Dixon-Coles MLE

**Question**: Should we replace the production Dixon-Coles MLE backbone with a hierarchical Bayesian Poisson model (PyMC / NumPyro)?

**Method**: Fit both models on EPL 2023-24, evaluate out-of-sample log-loss on EPL 2024-25, compare against the Pinnacle market ceiling.

**Status**: This notebook is a *companion* to `v11_phase0_mcmc_exploration.py`. The script runs the full analysis headless and writes `docs/v11_phase0_mcmc_report.md`. This notebook lets you re-run interactively + inspect posteriors visually.

## How to run

```bash
# Option A — headless (writes docs/v11_phase0_mcmc_report.md):
PYTHONPATH=apps/api/src .venv/bin/python notebooks/v11_phase0_mcmc_exploration.py

# Option B — interactive:
pip install jupyter
PYTHONPATH=apps/api/src jupyter notebook notebooks/v11_phase0_mcmc_exploration.ipynb
```

## 1. Imports + setup

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'notebooks'))
sys.path.insert(0, str(REPO_ROOT / 'apps' / 'api' / 'src'))

from v11_phase0_mcmc_exploration import (
    load_epl_season, build_dims,
    fit_mle, mh_sample, fit_map,
    unpack, compute_lambdas, predict_3way_from_lambdas,
    log_loss_3way, outcome_label, market_probs,
    effective_sample_size, r_hat,
)
print('Imports ok.')

## 2. Load EPL 2023-24 (train) + 2024-25 (test)

In [ ]:
train = load_epl_season('2324')
test  = load_epl_season('2425')
print(f'Train: {len(train)} matches  |  Test: {len(test)} matches')
train.head()

In [ ]:
# Distribution of goals — sanity check on the Poisson assumption
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(train['home_goals'], bins=range(0, 8), edgecolor='black', alpha=0.7)
axes[0].set(title='Home goals (train)', xlabel='Goals', ylabel='# matches')
axes[1].hist(train['away_goals'], bins=range(0, 8), edgecolor='black', alpha=0.7, color='C1')
axes[1].set(title='Away goals (train)', xlabel='Goals')
plt.tight_layout(); plt.show()
print('Mean home goals:', train['home_goals'].mean().round(2))
print('Mean away goals:', train['away_goals'].mean().round(2))

## 3. Build train arrays + fit DC MLE baseline

In [ ]:
dims = build_dims(train)
team_to_idx = {t: i for i, t in enumerate(dims.teams)}
home_idx = train['home'].map(team_to_idx).values
away_idx = train['away'].map(team_to_idx).values
hg = train['home_goals'].values
ag = train['away_goals'].values
print(f'{dims.n_teams} teams · {dims.n_params} params')

In [ ]:
%%time
mle_theta = fit_mle(dims, home_idx, away_idx, hg, ag)
home_adv_mle, attack_mle, defense_mle, rho_mle = unpack(mle_theta, dims)
print(f'MLE home_adv = {home_adv_mle:+.3f}  ·  rho = {rho_mle:+.4f}')

## 4. MCMC: hand-rolled Metropolis-Hastings

6 chains × 20k iter × 42 params. ~12s on a 2024 laptop.

In [ ]:
%%time
chains = mh_sample(dims, home_idx, away_idx, hg, ag,
                   n_chains=6, n_iter=20000, n_burn=5000)
print(f'Posterior samples shape: {chains.shape}')

## 5. Diagnostics: ESS + R-hat per parameter

In [ ]:
ess  = np.array([effective_sample_size(chains[:, :, p]) for p in range(dims.n_params)])
rhat = np.array([r_hat(chains[:, :, p]) for p in range(dims.n_params)])

fig, axes = plt.subplots(1, 2, figsize=(11, 3))
axes[0].bar(range(dims.n_params), ess, color='C0', alpha=0.7)
axes[0].axhline(100, color='red', linestyle='--', label='ESS = 100')
axes[0].set(title='Effective sample size per parameter', xlabel='Param index', ylabel='ESS')
axes[0].legend()
axes[1].bar(range(dims.n_params), rhat, color='C2', alpha=0.7)
axes[1].axhline(1.05, color='red', linestyle='--', label='R-hat = 1.05')
axes[1].axhline(1.10, color='orange', linestyle=':', label='R-hat = 1.10')
axes[1].set(title='R-hat per parameter', xlabel='Param index')
axes[1].legend()
plt.tight_layout(); plt.show()
print(f'Min ESS: {ess.min():.0f}  ·  Median ESS: {np.median(ess):.0f}')
print(f'Max R-hat: {rhat.max():.3f}  ·  Median R-hat: {np.median(rhat):.3f}')

In [ ]:
# Trace plot: home_adv (first param) across chains
fig, ax = plt.subplots(figsize=(10, 3))
for c in range(chains.shape[0]):
    ax.plot(chains[c, :, 0], alpha=0.6, label=f'chain {c}')
ax.set(title='Trace: home advantage parameter', xlabel='Iter (post burn-in)', ylabel='home_adv')
ax.legend(loc='upper right', fontsize=8)
plt.tight_layout(); plt.show()

## 6. Out-of-sample log-loss comparison

In [ ]:
seen = set(dims.teams)
test_clean = test[test['home'].isin(seen) & test['away'].isin(seen)].reset_index(drop=True)
h_idx_te = test_clean['home'].map(team_to_idx).values
a_idx_te = test_clean['away'].map(team_to_idx).values
outcomes_te = np.array([outcome_label(h, a) for h, a in zip(test_clean['home_goals'], test_clean['away_goals'])])
print(f'Test (after dropping promoted teams): {len(test_clean)}')

def predict_all(theta):
    ha, atk, dfn, rho = unpack(theta, dims)
    lh, la = compute_lambdas(h_idx_te, a_idx_te, ha, atk, dfn)
    probs = np.zeros((len(test_clean), 3))
    for i in range(len(test_clean)):
        probs[i] = predict_3way_from_lambdas(float(lh[i]), float(la[i]), rho)
    return probs

mle_probs  = predict_all(mle_theta)
posterior_mean = chains.reshape(-1, dims.n_params).mean(axis=0)
mcmc_probs = predict_all(posterior_mean)
market_p   = market_probs(test_clean['psh'].values, test_clean['psd'].values, test_clean['psa'].values)

results = pd.DataFrame({
    'model': ['Uniform', 'Pinnacle', 'DC MLE', 'MCMC mean'],
    'log_loss': [np.log(3), log_loss_3way(market_p, outcomes_te),
                 log_loss_3way(mle_probs, outcomes_te),
                 log_loss_3way(mcmc_probs, outcomes_te)],
})
results['delta_vs_market'] = results['log_loss'] - results.loc[1, 'log_loss']
results

## 7. Posterior of the home-advantage parameter

In [ ]:
flat = chains.reshape(-1, dims.n_params)
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(flat[:, 0], bins=60, alpha=0.7, color='C0', edgecolor='black')
ax.axvline(home_adv_mle, color='red', linestyle='--', linewidth=2, label=f'MLE = {home_adv_mle:+.3f}')
ax.axvline(flat[:, 0].mean(), color='C2', linestyle='-', linewidth=2, label=f'Posterior mean = {flat[:, 0].mean():+.3f}')
ax.set(title='Posterior: home advantage', xlabel='log home_adv', ylabel='Density')
ax.legend(); plt.tight_layout(); plt.show()

## 8. Verdict

See `docs/v11_phase0_mcmc_report.md` (auto-generated by the companion script) for the verdict text + recommendation paragraph.

Headline:
- MCMC outperforms DC MLE by ~6 milli-log-loss on this single season
- Convergence is borderline (R-hat ~1.06) — acceptable for exploration, not production
- Compute ratio ~450× makes the hand-rolled MH non-viable for production; NumPyro NUTS would be ~10× faster
- **Recommendation**: confirm on 2-3 more held-out seasons via NumPyro NUTS before paying the dependency cost